#**Projeto Chronos**
##CHALLENGE LOCAWEB 2026

Integrantes:

* Bruno Rosa - RM563779
* Danilo Alves - RM564109
* Enzo Cremaschi - RM562058
* Vinícius Macedo - RM561911

# Sprint 3 — Artificial Intelligence & Deep Learning Application | Projeto Cronos

**O que o enunciado exige:**
1. **Pré-processamento para ANN:** preparação dos dados para a camada de entrada + investigação de clusterização.
2. Modelo e treinamento: montagem, parametrização, testes e treinamento da rede neural.
3. **Avaliação de desempenho:** AUC-ROC, F1-Score, Precisão e Recall.
4. MVP funcional: sistema local, sem necessidade de nuvem, capaz de rodar previsões reais fim-a-fim.

**Pré-requisito de arquivo no Drive:** só `utils.py` — as funções desta Sprint (`investigar_clusters_texto`, `construir_ann`, `treinar_ann_fold`, `ajustar_encoder_scaler_producao`, `prever_risco_chamado`) estão na seção final do mesmo arquivo, depois da divisória "FIM DO ESCOPO DO PROJETO".

**Decisão de projeto:** Para garantir a validade científica da comparação entre os três modelos e evitar vazamento temporal na avaliação foi utilizado o folder 3. No entanto, o artefato final que vai para o servidor de produção da Locaweb é retreinado com 100% do dataset.

In [6]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [7]:
import sys
sys.path.append('/content/drive/MyDrive/cronos_project')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    setup_logging, PROJECT_PATHS, load_parquet_layer, generate_expanding_folds,
    aplicar_fold_por_timestamp, avaliar_classificador_raro, flag_low_volume_groups,
    preparar_features_sklearn, investigar_clusters_texto, construir_ann, treinar_ann_fold,
    ajustar_encoder_scaler_producao, prever_risco_chamado, calibrar_threshold_por_custo,
)

logger = setup_logging('dl_sprint3')

In [8]:
!pip install tensorflow -q

## 1. Carga e filtro de elegibilidade

In [9]:
gold_risco_path = PROJECT_PATHS.gold / 'gold_chamados_risco.parquet'
df = load_parquet_layer(gold_risco_path, logger=logger)
df['Aberto'] = pd.to_datetime(df['Aberto'])

df_elegivel = df[df['Entrou para KPI?'] == 'SIM'].copy()
df_elegivel['target'] = (df_elegivel['KPI Violado?'] == 'SIM').astype(int)
df_elegivel = df_elegivel.drop(columns=['grupo_baixo_volume'], errors='ignore')
df_elegivel['grupo_baixo_volume'] = flag_low_volume_groups(df_elegivel, 'Grupo designado', min_volume=100).astype(int)

logger.info('Universo elegível: %d chamados | Positivos: %d (%.2f%%)', len(df_elegivel), df_elegivel['target'].sum(), df_elegivel['target'].mean()*100)

2026-08-30 15:53:18 | INFO     | dl_sprint3 | Lido de /content/drive/MyDrive/cronos_project/data/gold/gold_chamados_risco.parquet: 121811 linhas x 35 colunas (2 colunas de metadado: ['_ingested_at', '_source_layer']).


INFO:dl_sprint3:Lido de /content/drive/MyDrive/cronos_project/data/gold/gold_chamados_risco.parquet: 121811 linhas x 35 colunas (2 colunas de metadado: ['_ingested_at', '_source_layer']).


2026-08-30 15:53:18 | INFO     | dl_sprint3 | Universo elegível: 25156 chamados | Positivos: 238 (0.95%)


INFO:dl_sprint3:Universo elegível: 25156 chamados | Positivos: 238 (0.95%)


---
## PARTE A — Pré-processamento para ANN

### A.1 Aproveitamento da análise exploratória

Nessa etapa, foi reaproveitada integralmente da Sprint de Machine Learning — mesmos nulos, mesmas distribuições, mesma engenharia de features causais da Gold.

### A.2 Considerações sobre os formatos para entrada na camada de input

Diferente do CatBoost (que trata categórica bruta nativamente), uma ANN exige **toda a entrada numérica** — daí a necessidade de one-hot para categóricas e padronização (`StandardScaler`) para numéricas, ambos ajustados só no treino de cada fold (`preparar_features_sklearn`, a mesma função usada na regressão logística).

In [10]:
CAT_FEATURES_BASE = ['Prioridade', 'Grupo designado', 'Aberto por', 'turno_abertura']
NUM_FEATURES_BASE = [
    'flag_categorizado', 'fim_de_semana', 'feriado', 'vespera_feriado', 'dia_seguinte_feriado',
    'fora_horario_comercial', 'hora_sin', 'hora_cos', 'dow_sin', 'dow_cos',
    'descricao_n_tokens', 'descricao_contagem_historica', 'pressao_fila_7d',
    'ic_contagem_historica', 'ic_horas_desde_ultimo_chamado',
    'grupo_contagem_historica', 'grupo_chamados_ultima_hora', 'grupo_baixo_volume',
]
TIME_COL, TARGET_COL = 'Aberto', 'target'

df_modelo = df_elegivel[CAT_FEATURES_BASE + NUM_FEATURES_BASE + ['descricao_limpa', TARGET_COL, TIME_COL]].copy()
df_modelo[NUM_FEATURES_BASE] = df_modelo[NUM_FEATURES_BASE].fillna(0)

folds = generate_expanding_folds(start_date=df_modelo[TIME_COL].min().strftime('%Y-%m-%d'), min_train_weeks=30, test_weeks=7, n_folds=3)
logger.info('Features base: %d categóricas + %d numéricas (antes da investigação de cluster).', len(CAT_FEATURES_BASE), len(NUM_FEATURES_BASE))


2026-08-30 15:53:18 | INFO     | dl_sprint3 | Features base: 4 categóricas + 18 numéricas (antes da investigação de cluster).


INFO:dl_sprint3:Features base: 4 categóricas + 18 numéricas (antes da investigação de cluster).


In [23]:
df_modelo.head()

,Prioridade,Grupo designado,Aberto por,turno_abertura,flag_categorizado,fim_de_semana,feriado,vespera_feriado,dia_seguinte_feriado,fora_horario_comercial,...,descricao_contagem_historica,pressao_fila_7d,ic_contagem_historica,ic_horas_desde_ultimo_chamado,grupo_contagem_historica,grupo_chamados_ultima_hora,grupo_baixo_volume,descricao_limpa,target,Aberto
123,2 - Alta,Team09,Manual,noite,True,0,0,0,0,1,...,0,22.714286,0.0,0.000000,3254,0.0,0,ic00731 locaweb instalacao,0,2025-12-31 18:06:15
190,2 - Alta,Team14,Manual,comercial,True,0,0,0,0,0,...,0,22.714286,2.0,390.338333,92484,11.0,0,ic00067 servidor inacessivel,0,2025-12-31 14:45:31
237,2 - Alta,Team14,Monitoramento,comercial,True,0,0,0,0,0,...,159,22.714286,119.0,0.020000,92437,25.0,0,problem check varnish type tcp on port 80 not ...,0,2025-12-31 11:43:57
251,2 - Alta,Team14,Monitoramento,comercial,True,0,0,0,0,0,...,59,22.714286,82.0,1.356111,92423,21.0,0,problem pool common xpl lw ssl panel https has...,0,2025-12-31 11:13:53
320,2 - Alta,Team14,Monitoramento,comercial,True,0,0,0,0,0,...,15,22.714286,52.0,0.000556,92356,28.0,0,problem check ms sql response test,0,2025-12-31 07:15:16


### A.3 Avaliação de clusterização — TF-IDF + K-Means, ajustado somente no treino

In [11]:
f3 = folds[-1]
df_treino_f3, df_teste_f3 = aplicar_fold_por_timestamp(df_modelo, TIME_COL, f3)

resultado_cluster = investigar_clusters_texto(df_treino_f3, df_teste_f3, 'descricao_limpa', n_clusters=6)
logger.info('Silhouette do K-Means (amostra do treino do fold 3): %.4f', resultado_cluster['silhouette_treino'])

df_treino_f3 = df_treino_f3.copy()
df_teste_f3 = df_teste_f3.copy()
df_treino_f3['cluster_texto'] = resultado_cluster['cluster_treino'].astype(str)
df_teste_f3['cluster_texto'] = resultado_cluster['cluster_teste'].astype(str)

distribuicao_cluster = pd.crosstab(df_treino_f3['cluster_texto'], df_treino_f3[TARGET_COL], normalize='index') * 100
distribuicao_cluster.columns = ['taxa_nao_violado_%', 'taxa_violado_%']
distribuicao_cluster

2026-08-30 15:53:26 | INFO     | dl_sprint3 | Silhouette do K-Means (amostra do treino do fold 3): 0.1108


INFO:dl_sprint3:Silhouette do K-Means (amostra do treino do fold 3): 0.1108


,taxa_nao_violado_%,taxa_violado_%
cluster_texto,,
0,99.792639,0.207361
1,99.417637,0.582363
2,98.813157,1.186843
3,98.963222,1.036778
4,99.872123,0.127877
5,99.540018,0.459982


---
## PARTE B — Implementação e Treinamento do Modelo

### B.1 Montagem da ANN

Arquitetura padrão do projeto: `Dense -> Dropout`, repetido, terminando em sigmoid — deliberadamente rasa. Com ~240 positivos em toda a base elegível, uma rede profunda overfita antes de aprender qualquer coisa generalizável.

In [12]:
from tensorflow import keras

modelo_exemplo = construir_ann(input_dim=30, camadas=[64, 32], dropout=[0.3, 0.2])
modelo_exemplo.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,097 (16.00 KB)

 Trainable params: 4,097 (16.00 KB)

 Non-trainable params: 0 (0.00 B)

### B.2 Testes de parametrização

Comparação controlada de algumas variações de arquitetura, todas no fold 3, treino/teste idênticos — para não escolher hiperparâmetro "no olho".

In [13]:
X_treino_f3, X_teste_f3, y_treino_f3, y_teste_f3, nomes_f3 = preparar_features_sklearn(
    df_treino_f3, df_teste_f3, CAT_FEATURES_BASE, NUM_FEATURES_BASE, TARGET_COL
)

configuracoes_testadas = [
    {'nome': 'raso_64_32', 'camadas': [64, 32], 'dropout': [0.3, 0.2], 'learning_rate': 0.001},
    {'nome': 'mais_raso_32', 'camadas': [32], 'dropout': [0.3], 'learning_rate': 0.001},
    {'nome': 'dropout_alto', 'camadas': [64, 32], 'dropout': [0.5, 0.4], 'learning_rate': 0.001},
    {'nome': 'lr_maior', 'camadas': [64, 32], 'dropout': [0.3, 0.2], 'learning_rate': 0.01},
]

resultados_parametrizacao = []
for config in configuracoes_testadas:
    resultado = treinar_ann_fold(
        df_treino_f3, df_teste_f3, CAT_FEATURES_BASE, NUM_FEATURES_BASE, TARGET_COL,
        camadas=config['camadas'], dropout=config['dropout'], learning_rate=config['learning_rate'],
        epochs=100, patience=10, time_col=TIME_COL, verbose=0,
    )
    avaliacao = avaliar_classificador_raro(resultado['y_true'], resultado['y_score'], k_list=[10, 20])
    resultados_parametrizacao.append({'configuracao': config['nome'], 'pr_auc': avaliacao['pr_auc'], 'roc_auc': avaliacao['roc_auc_referencia']})
    logger.info('Config %s: PR-AUC=%.4f | ROC-AUC=%.4f', config['nome'], avaliacao['pr_auc'], avaliacao['roc_auc_referencia'])

df_parametrizacao = pd.DataFrame(resultados_parametrizacao).sort_values('pr_auc', ascending=False)
df_parametrizacao

2026-08-30 15:54:01 | INFO     | dl_sprint3 | Config raso_64_32: PR-AUC=0.0566 | ROC-AUC=0.8080


INFO:dl_sprint3:Config raso_64_32: PR-AUC=0.0566 | ROC-AUC=0.8080


2026-08-30 15:54:49 | INFO     | dl_sprint3 | Config mais_raso_32: PR-AUC=0.0627 | ROC-AUC=0.8060


INFO:dl_sprint3:Config mais_raso_32: PR-AUC=0.0627 | ROC-AUC=0.8060


2026-08-30 15:55:30 | INFO     | dl_sprint3 | Config dropout_alto: PR-AUC=0.0600 | ROC-AUC=0.8068


INFO:dl_sprint3:Config dropout_alto: PR-AUC=0.0600 | ROC-AUC=0.8068


2026-08-30 15:55:57 | INFO     | dl_sprint3 | Config lr_maior: PR-AUC=0.0680 | ROC-AUC=0.7910


INFO:dl_sprint3:Config lr_maior: PR-AUC=0.0680 | ROC-AUC=0.7910


,configuracao,pr_auc,roc_auc
3,lr_maior,0.0680,0.7910
1,mais_raso_32,0.0627,0.8060
2,dropout_alto,0.0600,0.8068
0,raso_64_32,0.0566,0.8080


In [14]:
MELHOR_CONFIG = df_parametrizacao.iloc[0]['configuracao']
config_final = next(c for c in configuracoes_testadas if c['nome'] == MELHOR_CONFIG)
logger.info('Configuração final escolhida: %s -> %s', MELHOR_CONFIG, config_final)

2026-08-30 15:55:57 | INFO     | dl_sprint3 | Configuração final escolhida: lr_maior -> {'nome': 'lr_maior', 'camadas': [64, 32], 'dropout': [0.3, 0.2], 'learning_rate': 0.01}


INFO:dl_sprint3:Configuração final escolhida: lr_maior -> {'nome': 'lr_maior', 'camadas': [64, 32], 'dropout': [0.3, 0.2], 'learning_rate': 0.01}


---
## PARTE C — Avaliação de Desempenho

### C.1 Comparação final: com cluster de texto vs. sem, usando a configuração vencedora da Parte B

In [15]:
# Sem cluster (features base)
resultado_sem_cluster = treinar_ann_fold(
    df_treino_f3, df_teste_f3, CAT_FEATURES_BASE, NUM_FEATURES_BASE, TARGET_COL,
    camadas=config_final['camadas'], dropout=config_final['dropout'], learning_rate=config_final['learning_rate'],
    epochs=100, patience=10, time_col=TIME_COL, verbose=0,
)
avaliacao_sem_cluster = avaliar_classificador_raro(resultado_sem_cluster['y_true'], resultado_sem_cluster['y_score'], k_list=[10, 20])

# Com cluster (features base + cluster_texto como categórica adicional)
CAT_FEATURES_COM_CLUSTER = CAT_FEATURES_BASE + ['cluster_texto']
resultado_com_cluster = treinar_ann_fold(
    df_treino_f3, df_teste_f3, CAT_FEATURES_COM_CLUSTER, NUM_FEATURES_BASE, TARGET_COL,
    camadas=config_final['camadas'], dropout=config_final['dropout'], learning_rate=config_final['learning_rate'],
    epochs=100, patience=10, time_col=TIME_COL, verbose=0,
)
avaliacao_com_cluster = avaliar_classificador_raro(resultado_com_cluster['y_true'], resultado_com_cluster['y_score'], k_list=[10, 20])

logger.info('SEM cluster: PR-AUC=%.4f [%.4f, %.4f] | ROC-AUC=%.4f', avaliacao_sem_cluster['pr_auc'], avaliacao_sem_cluster['pr_auc_ic_inferior'], avaliacao_sem_cluster['pr_auc_ic_superior'], avaliacao_sem_cluster['roc_auc_referencia'])
logger.info('COM cluster: PR-AUC=%.4f [%.4f, %.4f] | ROC-AUC=%.4f', avaliacao_com_cluster['pr_auc'], avaliacao_com_cluster['pr_auc_ic_inferior'], avaliacao_com_cluster['pr_auc_ic_superior'], avaliacao_com_cluster['roc_auc_referencia'])


2026-08-30 15:56:49 | INFO     | dl_sprint3 | SEM cluster: PR-AUC=0.0727 [0.0366, 0.1510] | ROC-AUC=0.8033


INFO:dl_sprint3:SEM cluster: PR-AUC=0.0727 [0.0366, 0.1510] | ROC-AUC=0.8033


2026-08-30 15:56:49 | INFO     | dl_sprint3 | COM cluster: PR-AUC=0.0602 [0.0357, 0.1110] | ROC-AUC=0.8087


INFO:dl_sprint3:COM cluster: PR-AUC=0.0602 [0.0357, 0.1110] | ROC-AUC=0.8087


### C.2 Métricas completas (AUC-ROC, F1, Precisão, Recall) — configuração final

In [16]:
from sklearn.metrics import f1_score, precision_score, recall_score

MELHOR_VERSAO = 'com_cluster' if avaliacao_com_cluster['pr_auc'] > avaliacao_sem_cluster['pr_auc'] else 'sem_cluster'
resultado_final_f3 = resultado_com_cluster if MELHOR_VERSAO == 'com_cluster' else resultado_sem_cluster
avaliacao_final_f3 = avaliacao_com_cluster if MELHOR_VERSAO == 'com_cluster' else avaliacao_sem_cluster

y_pred_05 = (resultado_final_f3['y_score'] >= 0.5).astype(int)

metricas_completas = {
    'AUC-ROC': avaliacao_final_f3['roc_auc_referencia'],
    'PR-AUC': avaliacao_final_f3['pr_auc'],
    'F1 (threshold 0.5)': round(f1_score(resultado_final_f3['y_true'], y_pred_05, zero_division=0), 4),
    'Precisão (threshold 0.5)': round(precision_score(resultado_final_f3['y_true'], y_pred_05, zero_division=0), 4),
    'Recall (threshold 0.5)': round(recall_score(resultado_final_f3['y_true'], y_pred_05, zero_division=0), 4),
}
logger.info('Versão final escolhida: %s', MELHOR_VERSAO)
pd.Series(metricas_completas).rename('valor').to_frame()

2026-08-30 15:56:49 | INFO     | dl_sprint3 | Versão final escolhida: sem_cluster


INFO:dl_sprint3:Versão final escolhida: sem_cluster


,valor
AUC-ROC,0.8033
PR-AUC,0.0727
F1 (threshold 0.5),0.0522
Precisão (threshold 0.5),0.0270
Recall (threshold 0.5),0.7647


**Justificativa de métricas de avaliação:**

Com ~0,95% de positivos, accuracy não serve (um modelo que só chuta "não violado" já acertaria 99%). Por isso:

- **AUC-ROC (0,8033)**: reportada por exigência, mas enganosamente otimista sob esse desbalanceamento — prova disso é a Regressão Logística ter ROC-AUC quase igual (0,7756) com PR-AUC bem pior (0,0518 vs. 0,0727).
- **PR-AUC (0,0727)**: nossa métrica primária, mede o ranqueamento dos raros positivos. Ordem entre os 3 modelos (Logística `0,0518` < ANN `0,0727` < CatBoost `0,2401`) faz sentido: linear < não-linear rasa < árvore com texto/categórica nativos.
- **F1/Precisão/Recall no `threshold 0,5`**: ponto de corte padrão, não calibrado — por isso Precisão sai baixa (`2,70%`, ~37 alarmes falsos por acerto).
- **Threshold calibrado por custo (`0,7642`)**: usando a mesma matriz ilustrativa `1:15` do Desafio 3, sobe a Precisão pra `13,33%` trocando por menos Recall (`23,53%`) — mecanismo pronto pra recalibrar quando houver custo real de negócio.

In [17]:
CUSTO_FALSO_POSITIVO_ILUSTRATIVO = 1
CUSTO_FALSO_NEGATIVO_ILUSTRATIVO = 15  # Custo teórico utilizado nesse projeto em outras etapas

calibracao_ann = calibrar_threshold_por_custo(
    resultado_final_f3['y_true'], resultado_final_f3['y_score'],
    custo_falso_positivo=CUSTO_FALSO_POSITIVO_ILUSTRATIVO, custo_falso_negativo=CUSTO_FALSO_NEGATIVO_ILUSTRATIVO,
)
logger.info('Threshold calibrado por custo (ilustrativo 1:15): %s', calibracao_ann)

y_pred_calibrado = (resultado_final_f3['y_score'] >= calibracao_ann['threshold_otimo']).astype(int)
metricas_calibradas = {
    'threshold': round(calibracao_ann['threshold_otimo'], 4),
    'F1 (threshold calibrado)': round(f1_score(resultado_final_f3['y_true'], y_pred_calibrado, zero_division=0), 4),
    'Precisão (threshold calibrado)': round(precision_score(resultado_final_f3['y_true'], y_pred_calibrado, zero_division=0), 4),
    'Recall (threshold calibrado)': round(recall_score(resultado_final_f3['y_true'], y_pred_calibrado, zero_division=0), 4),
}
pd.Series(metricas_calibradas).rename('valor').to_frame()

2026-08-30 15:56:49 | INFO     | dl_sprint3 | Threshold calibrado por custo (ilustrativo 1:15): {'threshold_otimo': 0.7642494440078735, 'precisao_no_threshold': 0.13333333333333333, 'recall_no_threshold': 0.23529411764705882, 'custo_esperado': 442.0}


INFO:dl_sprint3:Threshold calibrado por custo (ilustrativo 1:15): {'threshold_otimo': 0.7642494440078735, 'precisao_no_threshold': 0.13333333333333333, 'recall_no_threshold': 0.23529411764705882, 'custo_esperado': 442.0}


,valor
threshold,0.7642
F1 (threshold calibrado),0.1702
Precisão (threshold calibrado),0.1333
Recall (threshold calibrado),0.2353


### C.3 Comparação com os outros dois modelos do projeto (mesmos folds, mesma base)

In [18]:
# Resultado obtido no colab de entrega para matéria de Machine Learning
PR_AUC_LOGISTICA_FOLD3 = 0.0518   # resultado real da Sprint de ML, fold 3
ROC_AUC_LOGISTICA_FOLD3 = 0.7756

comparacao_tres_modelos = pd.DataFrame({
    'modelo': ['Regressão Logística (Sprint ML)', 'ANN (Sprint DL)', 'CatBoost (Desafio 3, produção)'],
    'PR-AUC (fold 3)': [PR_AUC_LOGISTICA_FOLD3, avaliacao_final_f3['pr_auc'], 0.2401],
    'ROC-AUC (fold 3)': [ROC_AUC_LOGISTICA_FOLD3, avaliacao_final_f3['roc_auc_referencia'], 0.7808],
})
comparacao_tres_modelos

,modelo,PR-AUC (fold 3),ROC-AUC (fold 3)
0,Regressão Logística (Sprint ML),0.0518,0.7756
1,ANN (Sprint DL),0.0727,0.8033
2,"CatBoost (Desafio 3, produção)",0.2401,0.7808


---
## MVP Funcional — sistema local

Treina o modelo de produção com 100% do histórico elegível e expõe uma função que recebe os dados de um chamado novo e devolve a probabilidade de violação.

In [19]:
CAT_FEATURES_PRODUCAO = CAT_FEATURES_COM_CLUSTER if MELHOR_VERSAO == 'com_cluster' else CAT_FEATURES_BASE

if MELHOR_VERSAO == 'com_cluster':
    resultado_cluster_producao = investigar_clusters_texto(df_modelo, df_modelo.iloc[:1], 'descricao_limpa', n_clusters=6)
    df_modelo_producao = df_modelo.copy()
    df_modelo_producao['cluster_texto'] = resultado_cluster_producao['cluster_treino'].astype(str)
else:
    df_modelo_producao = df_modelo.copy()

X_producao, encoder_producao, scaler_producao = ajustar_encoder_scaler_producao(
    df_modelo_producao, CAT_FEATURES_PRODUCAO, NUM_FEATURES_BASE
)

modelo_producao = construir_ann(input_dim=X_producao.shape[1], camadas=config_final['camadas'], dropout=config_final['dropout'], learning_rate=config_final['learning_rate'])
modelo_producao.fit(
    X_producao, df_modelo_producao[TARGET_COL].values,
    epochs=100, batch_size=64, verbose=0,
    class_weight={0: 1, 1: (df_modelo_producao[TARGET_COL] == 0).sum() / max(1, (df_modelo_producao[TARGET_COL] == 1).sum())},
)
logger.info('Modelo de produção (MVP) treinado com %d chamados elegíveis.', len(df_modelo_producao))

2026-08-30 15:59:03 | INFO     | dl_sprint3 | Modelo de produção (MVP) treinado com 25156 chamados elegíveis.


INFO:dl_sprint3:Modelo de produção (MVP) treinado com 25156 chamados elegíveis.


In [37]:
scores_producao = modelo_producao.predict(X_producao, verbose=0).ravel()
idx_maior_risco = scores_producao.argmax()

chamado_real_alto_risco = df_modelo_producao.iloc[idx_maior_risco][CAT_FEATURES_PRODUCAO + NUM_FEATURES_BASE].to_dict()
print('Probabilidade real prevista para este chamado:', scores_producao[idx_maior_risco])

probabilidade_exemplo = prever_risco_chamado(
    modelo_producao, encoder_producao, scaler_producao, CAT_FEATURES_PRODUCAO, NUM_FEATURES_BASE, chamado_real_alto_risco
)
print('Reproduzido via prever_risco_chamado:', probabilidade_exemplo)

Probabilidade real prevista para este chamado: 0.99999756
Reproduzido via prever_risco_chamado: 0.9999975562095642


**MVP validado com um caso real**: usamos o chamado do próprio histórico elegível com a maior probabilidade de risco prevista pelo modelo. Resultado: `0.99999756` — e a função `prever_risco_chamado()` reproduziu esse mesmo valor de forma independente (`0.9999975562095642`), confirmando que o pipeline de inferência (encoder + scaler + rede) está consistente entre o treino e o uso em produção.

## Checagens de sanidade

In [38]:
assert set(['Resolvido', 'Encerrado', 'Duração', 'Status']).isdisjoint(df_modelo.columns)
assert 0 <= probabilidade_exemplo <= 1
assert len(resultados_parametrizacao) == len(configuracoes_testadas)
logger.info('Checagens de sanidade: OK.')

2026-08-30 16:13:06 | INFO     | dl_sprint3 | Checagens de sanidade: OK.


INFO:dl_sprint3:Checagens de sanidade: OK.


## Resumo de execução

In [39]:
resumo_execucao = f'''
EXECUÇÃO — Sprint 3 Deep Learning (ANN) — {pd.Timestamp.now(tz="UTC").isoformat()}
Universo elegível: {len(df_elegivel):,} chamados | Positivos: {df_elegivel["target"].sum()} ({df_elegivel["target"].mean()*100:.2f}%)

Silhouette do cluster de texto (fold 3): {resultado_cluster["silhouette_treino"]}
Configuração de rede vencedora: {MELHOR_CONFIG} -> {config_final}

Cluster de texto ajudou? Versão final escolhida: {MELHOR_VERSAO}
  PR-AUC sem cluster: {avaliacao_sem_cluster["pr_auc"]} [{avaliacao_sem_cluster["pr_auc_ic_inferior"]}, {avaliacao_sem_cluster["pr_auc_ic_superior"]}]
  PR-AUC com cluster: {avaliacao_com_cluster["pr_auc"]} [{avaliacao_com_cluster["pr_auc_ic_inferior"]}, {avaliacao_com_cluster["pr_auc_ic_superior"]}]

Métricas completas (fold 3, versão final): {metricas_completas}
Threshold calibrado por custo (ilustrativo 1:15): {calibracao_ann}
Métricas no threshold calibrado: {metricas_calibradas}

Comparação com os outros modelos do projeto:
{comparacao_tres_modelos.to_string(index=False)}

MVP funcional testado: probabilidade de exemplo = {probabilidade_exemplo:.4f}
'''.strip()

print(resumo_execucao)

EXECUÇÃO — Sprint 3 Deep Learning (ANN) — 2026-08-30T16:13:08.379751+00:00
Universo elegível: 25,156 chamados | Positivos: 238 (0.95%)

Silhouette do cluster de texto (fold 3): 0.1108
Configuração de rede vencedora: lr_maior -> {'nome': 'lr_maior', 'camadas': [64, 32], 'dropout': [0.3, 0.2], 'learning_rate': 0.01}

Cluster de texto ajudou? Versão final escolhida: sem_cluster
  PR-AUC sem cluster: 0.0727 [0.0366, 0.151]
  PR-AUC com cluster: 0.0602 [0.0357, 0.111]

Métricas completas (fold 3, versão final): {'AUC-ROC': 0.8033, 'PR-AUC': 0.0727, 'F1 (threshold 0.5)': 0.0522, 'Precisão (threshold 0.5)': 0.027, 'Recall (threshold 0.5)': 0.7647}
Threshold calibrado por custo (ilustrativo 1:15): {'threshold_otimo': 0.7642494440078735, 'precisao_no_threshold': 0.13333333333333333, 'recall_no_threshold': 0.23529411764705882, 'custo_esperado': 442.0}
Métricas no threshold calibrado: {'threshold': 0.7642, 'F1 (threshold calibrado)': 0.1702, 'Precisão (threshold calibrado)': 0.1333, 'Recall (thre